# Análisis Exploratorio de Datos de Statsbomb

Este notebook permite explorar los datos procesados por el sistema de streaming.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Cargar Datos Procesados

In [ ]:
# Rutas a los datos
DATA_PATH = Path('/app/data/processed')

# Cargar estadísticas de posesión
try:
    possession_df = pd.read_parquet(DATA_PATH / 'possession_stats')
    print(f"Posesión: {len(possession_df)} registros cargados")
    display(possession_df.head())
except Exception as e:
    print(f"Error cargando datos de posesión: {e}")

In [ ]:
# Cargar estadísticas de xG
try:
    xg_df = pd.read_parquet(DATA_PATH / 'xg_stats')
    print(f"xG: {len(xg_df)} registros cargados")
    display(xg_df.head())
except Exception as e:
    print(f"Error cargando datos de xG: {e}")

In [ ]:
# Cargar estadísticas de pases
try:
    pass_df = pd.read_parquet(DATA_PATH / 'pass_stats')
    print(f"Pases: {len(pass_df)} registros cargados")
    display(pass_df.head())
except Exception as e:
    print(f"Error cargando datos de pases: {e}")

## 2. Visualización de Posesión

In [ ]:
if 'possession_df' in locals() and not possession_df.empty:
    plt.figure(figsize=(14, 6))
    
    for team in possession_df['team_name'].unique():
        team_data = possession_df[possession_df['team_name'] == team]
        plt.plot(team_data['window_start'], 
                team_data['possession_percentage'], 
                marker='o', label=team, linewidth=2)
    
    plt.xlabel('Tiempo', fontsize=12)
    plt.ylabel('Posesión (%)', fontsize=12)
    plt.title('Evolución de la Posesión por Equipo', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 3. Análisis de Expected Goals (xG)

In [ ]:
if 'xg_df' in locals() and not xg_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # xG total por equipo
    xg_by_team = xg_df.groupby('team_name')['total_xg'].sum().sort_values(ascending=False)
    axes[0].barh(xg_by_team.index, xg_by_team.values, color='skyblue')
    axes[0].set_xlabel('xG Total', fontsize=12)
    axes[0].set_title('Expected Goals Total por Equipo', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='x')
    
    # Tiros por equipo
    shots_by_team = xg_df.groupby('team_name')['shots_count'].sum().sort_values(ascending=False)
    axes[1].barh(shots_by_team.index, shots_by_team.values, color='coral')
    axes[1].set_xlabel('Número de Tiros', fontsize=12)
    axes[1].set_title('Total de Tiros por Equipo', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

## 4. Análisis de Pases

In [ ]:
if 'pass_df' in locals() and not pass_df.empty:
    plt.figure(figsize=(14, 6))
    
    for team in pass_df['team_name'].unique():
        team_data = pass_df[pass_df['team_name'] == team]
        plt.plot(team_data['window_start'], 
                team_data['pass_completion_pct'], 
                marker='s', label=team, linewidth=2)
    
    plt.xlabel('Tiempo', fontsize=12)
    plt.ylabel('Tasa de Éxito en Pases (%)', fontsize=12)
    plt.title('Evolución de la Precisión de Pases por Equipo', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Cargar y Analizar Modelo ML

In [ ]:
import joblib

MODEL_PATH = Path('/app/data/models')

# Cargar modelo
try:
    model = joblib.load(MODEL_PATH / 'modelo.joblib')
    scaler = joblib.load(MODEL_PATH / 'scaler.joblib')
    metadata = joblib.load(MODEL_PATH / 'model_metadata.joblib')
    
    print("Modelo cargado exitosamente!")
    print(f"\nTipo: {metadata['model_type']}")
    print(f"Clases: {metadata['classes']}")
    print(f"Fecha de entrenamiento: {metadata['timestamp']}")
except Exception as e:
    print(f"Error cargando modelo: {e}")

## 6. Feature Importance (si es Random Forest)

In [ ]:
if 'model' in locals() and hasattr(model, 'feature_importances_'):
    feature_names = [
        'home_possession_count', 'away_possession_count',
        'home_shots', 'away_shots',
        'home_xg', 'away_xg',
        'home_passes', 'away_passes',
        'home_completed_passes', 'away_completed_passes',
        'home_pass_completion', 'away_pass_completion',
        'home_duels', 'away_duels',
        'home_tackles', 'away_tackles'
    ]
    
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(12, 8))
    plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
    plt.xlabel('Importancia', fontsize=12)
    plt.title('Feature Importance - Random Forest', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

## 7. Estadísticas de Rendimiento del Sistema

Aquí puedes agregar métricas de rendimiento capturadas de la Spark UI.

In [ ]:
# Ejemplo de comparación de rendimiento
performance_data = {
    'Métrica': [
        'Tiempo Total (s)',
        'Avg Task Time (s)',
        'Shuffle Read (MB)',
        'Shuffle Write (MB)',
        'GC Time (s)',
        'Memory Spill (MB)',
    ],
    'Intel i7 / 32GB': [0, 0, 0, 0, 0, 0],  # Llenar con datos reales
    'Intel i3 / 8GB': [0, 0, 0, 0, 0, 0]    # Llenar con datos reales
}

perf_df = pd.DataFrame(performance_data)
display(perf_df)